# Loading the E-commerce Dataset into MySQL

This notebook is the **setup step**. It reads the seven CSVs downloaded from Kaggle, infers a
SQL column type for every pandas dtype, creates a matching MySQL table for each file, and
bulk-inserts the rows with `executemany` in batches of 5,000.

Run this before [`Questions.ipynb`](Questions.ipynb), which does the actual analysis.

**Before running:**

1. Download the dataset (link in `Ecommerce_Dataset_Link.txt`) and unzip the seven CSVs into a folder.
2. Create the target database in MySQL: `CREATE DATABASE Ecommerce;`
3. Copy `.env.example` to `.env` and set your credentials and `DATA_DIR`.
4. Install dependencies: `pip install -r requirements.txt`

**Re-running is safe.** Each table is checked for existing rows before anything is inserted, and
skipped if it already holds data — the inserts are unconditional, so without that check a second
run would duplicate the entire dataset. To genuinely reload, drop the tables (or the database)
first.


In [1]:
import os

import mysql.connector
import pandas as pd
from dotenv import load_dotenv

# Credentials and paths are read from a local .env file that is never committed.
# Copy .env.example to .env and fill in your own values.
load_dotenv()

conn = mysql.connector.connect(
    host=os.getenv("MYSQL_HOST", "localhost"),
    user=os.getenv("MYSQL_USER", "root"),
    password=os.getenv("MYSQL_PASSWORD"),
    database=os.getenv("MYSQL_DATABASE", "Ecommerce"),
)
cursor = conn.cursor()

# Folder holding the seven CSVs unzipped from the Kaggle download.
folder_path = os.getenv("DATA_DIR", "./Ecommerce_data")

csv_files = [
    ("customers.csv", "customers"),
    ("orders.csv", "orders"),
    ("sellers.csv", "sellers"),
    ("products.csv", "products"),
    ("geolocation.csv", "geolocation"),
    ("payments.csv", "payments"),
    ("order_items.csv", "order_items"),
]

BATCH_SIZE = 5_000


def get_sql_type(dtype):
    """Map a pandas dtype onto the MySQL column type used to create the table."""
    if pd.api.types.is_integer_dtype(dtype):
        return "INT"
    elif pd.api.types.is_float_dtype(dtype):
        return "FLOAT"
    elif pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "DATETIME"
    else:
        return "TEXT"


for csv_file, table_name in csv_files:
    df = pd.read_csv(os.path.join(folder_path, csv_file))

    # Column names that are safe to use as SQL identifiers.
    df.columns = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in df.columns]

    columns = ", ".join(f"`{c}` {get_sql_type(df[c].dtype)}" for c in df.columns)
    cursor.execute(f"CREATE TABLE IF NOT EXISTS `{table_name}` ({columns})")

    # Guard against a second run. The inserts below are unconditional, so loading into
    # a table that already holds rows silently duplicates the whole dataset -- which is
    # exactly how this database once ended up with six copies of every order.
    cursor.execute(f"SELECT COUNT(*) FROM `{table_name}`")
    existing = cursor.fetchone()[0]
    if existing:
        print(f"{table_name:<12} SKIPPED - already holds {existing:,} rows")
        continue

    # NaN -> None so MySQL stores a real NULL. This needs an object dtype: on a float
    # column, NaN survives .where() unchanged.
    rows = list(df.astype(object).where(pd.notnull(df), None).itertuples(index=False, name=None))

    col_list = ", ".join(f"`{c}`" for c in df.columns)
    placeholders = ", ".join(["%s"] * len(df.columns))
    sql = f"INSERT INTO `{table_name}` ({col_list}) VALUES ({placeholders})"

    for start in range(0, len(rows), BATCH_SIZE):
        cursor.executemany(sql, rows[start : start + BATCH_SIZE])
    conn.commit()

    nulls = int(df.isnull().sum().sum())
    print(f"{table_name:<12} loaded {len(rows):>9,} rows  ({nulls:,} nulls)")

conn.close()
print("\nDone.")


customers    loaded    99,441 rows  (0 nulls)
orders       loaded    99,441 rows  (4,908 nulls)
sellers      loaded     3,095 rows  (0 nulls)
products     loaded    32,951 rows  (2,448 nulls)
geolocation  loaded 1,000,163 rows  (0 nulls)
payments     loaded   103,886 rows  (0 nulls)
order_items  loaded   112,650 rows  (0 nulls)

Done.
